# Library Management System

**Lumexa Python Foundations · Course 2: Problem Solving & Program Design · Lessons 4–5**

A library system that manages books and borrowing: search, sorting, loans and tests. In Lesson 8 you rebuild it with classes.

* **Part 1 · Lesson 4:** searching, sorting, counting and filtering a catalog of books.
* **Part 2 · Lesson 5:** borrowing and returning books, checked with tests (`assert`) and debugging.

▶ Run the code cell below (**Shift + Enter**) and type your answers when the program asks. To keep your own copy, choose **File → Save a copy in Drive**.

In [ ]:
# ======================================================================
#  LIBRARY MANAGEMENT SYSTEM
#  Lumexa Python Foundations · Course 2: Problem Solving & Program Design
#  Part 1 · Lesson 4: algorithms and problem-solving patterns
#  Part 2 · Lesson 5: debugging, testing and fixing problems
# ======================================================================


# ----------------------------------------------------------------------
# PART 1 · Lesson 4: a catalog you can search, sort, count and filter
# ----------------------------------------------------------------------
# Every book is a record (a dictionary) and the catalog is a list of records.
books = [
    {"id": 1, "title": "The Martian", "author": "Andy Weir", "year": 2011, "genre": "Science Fiction", "available": True},
    {"id": 2, "title": "Dune", "author": "Frank Herbert", "year": 1965, "genre": "Science Fiction", "available": True},
    {"id": 3, "title": "Cosmos", "author": "Carl Sagan", "year": 1980, "genre": "Science", "available": True},
    {"id": 4, "title": "A Brief History of Time", "author": "Stephen Hawking", "year": 1988, "genre": "Science", "available": True},
    {"id": 5, "title": "The Hobbit", "author": "J.R.R. Tolkien", "year": 1937, "genre": "Fantasy", "available": True},
    {"id": 6, "title": "Charlotte's Web", "author": "E.B. White", "year": 1952, "genre": "Fiction", "available": True},
    {"id": 7, "title": "Hidden Figures", "author": "Margot Lee Shetterly", "year": 2016, "genre": "History", "available": True},
    {"id": 8, "title": "The Little Prince", "author": "Antoine de Saint-Exupery", "year": 1943, "genre": "Fiction", "available": True},
]


def show_books(book_list, heading):
    """Print a list of books as a neat table."""
    print()
    print(heading)
    print(f"{'ID':<4}{'Title':<28}{'Author':<27}{'Year':<6}Status")
    for book in book_list:
        if book["available"]:
            status = "available"
        else:
            status = "ON LOAN"
        print(f"{book['id']:<4}{book['title']:<28}{book['author']:<27}{book['year']:<6}{status}")
    print(f"({len(book_list)} book(s))")


def find_book(book_list, book_id):
    """Linear search by id: look at every book until we find it. Work grows in a linear way with the list."""
    for book in book_list:
        if book["id"] == book_id:
            return book
    return None


def search(book_list, text):
    """Linear search by title or author. Partial matches work and capital letters do not matter."""
    matches = []
    for book in book_list:
        if text.lower() in book["title"].lower() or text.lower() in book["author"].lower():
            matches.append(book)
    return matches


def get_title(book):
    return book["title"]


def get_year(book):
    return book["year"]


def oldest_and_newest(book_list):
    """The accumulator pattern: keep the best so far while looping once through the list."""
    oldest = book_list[0]
    newest = book_list[0]
    for book in book_list:
        if book["year"] < oldest["year"]:
            oldest = book
        if book["year"] > newest["year"]:
            newest = book
    return oldest, newest


def count_by_genre(book_list):
    """A frequency table: a dictionary that counts how many books each genre has."""
    counts = {}
    for book in book_list:
        genre = book["genre"]
        counts[genre] = counts.get(genre, 0) + 1
    return counts


def available_books(book_list):
    """Filtering: keep only the books that are on the shelf."""
    return [book for book in book_list if book["available"]]


show_books(sorted(books, key=get_title), "CATALOG, sorted by title")
show_books(sorted(books, key=get_year, reverse=True), "CATALOG, newest first")
show_books(search(books, "carl"), "SEARCH: 'carl'")
show_books(search(books, "THE "), "SEARCH: 'THE '")

oldest, newest = oldest_and_newest(books)
print()
print("Oldest book:", oldest["title"], f"({oldest['year']})")
print("Newest book:", newest["title"], f"({newest['year']})")
print("Books per genre:")
genre_counts = count_by_genre(books)
for genre in sorted(genre_counts):
    print(f"  {genre}: {genre_counts[genre]}")
print("Books on the shelf:", len(available_books(books)))


# ----------------------------------------------------------------------
# PART 2 · Lesson 5: borrowing and returning, tested before we trust it
# ----------------------------------------------------------------------
LOAN_LIMIT = 3
DEBUG = False          # set to True to print the inner steps (print debugging)

members = {
    101: {"name": "Nova", "borrowed": []},
    102: {"name": "Orion", "borrowed": []},
}


def debug(*messages):
    """Print only while DEBUG is True, so the debugging prints can stay in the code."""
    if DEBUG:
        print("[debug]", *messages)


def borrow_book(book_list, member_dict, member_id, book_id):
    """Lend a book. Returns (True or False, a message). Every refusal has its own message."""
    debug("borrow_book", member_id, book_id)
    member = member_dict.get(member_id)
    if member is None:
        return False, f"There is no member {member_id}."
    book = find_book(book_list, book_id)
    if book is None:
        return False, f"There is no book {book_id}."
    if not book["available"]:
        return False, f"'{book['title']}' is already out."
    if len(member["borrowed"]) >= LOAN_LIMIT:
        return False, f"{member['name']} already has {LOAN_LIMIT} books."
    book["available"] = False
    member["borrowed"].append(book_id)
    debug("loans of", member["name"], "are now", member["borrowed"])
    return True, f"{member['name']} borrowed '{book['title']}'."


def return_book(book_list, member_dict, member_id, book_id):
    """Take a book back. Returns (True or False, a message)."""
    debug("return_book", member_id, book_id)
    member = member_dict.get(member_id)
    if member is None:
        return False, f"There is no member {member_id}."
    book = find_book(book_list, book_id)
    if book is None:
        return False, f"There is no book {book_id}."
    if book_id not in member["borrowed"]:
        return False, f"{member['name']} did not borrow '{book['title']}'."
    book["available"] = True
    member["borrowed"].remove(book_id)
    return True, f"{member['name']} returned '{book['title']}'."


# --- Tests: each one is a normal case, an edge case or an invalid case with a result we expect. ---
def fresh_books():
    return [
        {"id": 1, "title": "Dune", "author": "Frank Herbert", "year": 1965, "genre": "Science Fiction", "available": True},
        {"id": 2, "title": "Cosmos", "author": "Carl Sagan", "year": 1980, "genre": "Science", "available": True},
        {"id": 3, "title": "The Hobbit", "author": "J.R.R. Tolkien", "year": 1937, "genre": "Fantasy", "available": True},
        {"id": 4, "title": "Hidden Figures", "author": "Margot Lee Shetterly", "year": 2016, "genre": "History", "available": True},
    ]


def fresh_members():
    return {1: {"name": "Ana", "borrowed": []}, 2: {"name": "Ben", "borrowed": []}}


test_results = []


def check(description, condition):
    test_results.append((description, condition))
    if condition:
        print("  PASS", description)
    else:
        print("  FAIL", description)


print()
print("RUNNING TESTS")
shelf = fresh_books()
people = fresh_members()

ok, message = borrow_book(shelf, people, 1, 1)
check("normal: an available book can be borrowed", ok and shelf[0]["available"] is False and people[1]["borrowed"] == [1])
ok, message = borrow_book(shelf, people, 2, 1)
check("invalid: a book that is already out is refused", ok is False and "already out" in message)
ok, message = borrow_book(shelf, people, 99, 2)
check("invalid: an unknown member is refused", ok is False and "no member" in message)
ok, message = borrow_book(shelf, people, 1, 99)
check("invalid: an unknown book is refused", ok is False and "no book" in message)
ok, message = return_book(shelf, people, 2, 1)
check("invalid: returning a book you did not borrow is refused", ok is False and "did not borrow" in message)
ok, message = return_book(shelf, people, 1, 1)
check("normal: a borrowed book can be returned", ok and shelf[0]["available"] is True and people[1]["borrowed"] == [])
borrow_book(shelf, people, 1, 1)
borrow_book(shelf, people, 1, 2)
borrow_book(shelf, people, 1, 3)
ok, message = borrow_book(shelf, people, 1, 4)
check("edge: the fourth book is refused when the limit is 3", ok is False and "already has 3" in message and shelf[3]["available"] is True)
check("search ignores capital letters", len(search(shelf, "DUNE")) == 1)
check("search finds nothing for a missing word", search(shelf, "zebra") == [])
check("sorting by year puts the oldest first", sorted(shelf, key=get_year)[0]["title"] == "The Hobbit")
check("oldest and newest are found", oldest_and_newest(shelf)[0]["year"] == 1937 and oldest_and_newest(shelf)[1]["year"] == 2016)
check("the genre counts add up to the number of books", sum(count_by_genre(shelf).values()) == len(shelf))

failed = [description for description, condition in test_results if not condition]
print(f"{len(test_results) - len(failed)} of {len(test_results)} tests passed")
assert len(failed) == 0, "Fix these first: " + ", ".join(failed)      # stop right here if any test fails

# Print debugging: switch DEBUG on to watch one borrow step by step, then switch it off again.
print()
print("DEBUG DEMO")
DEBUG = True
borrow_book(books, members, 101, 5)
return_book(books, members, 101, 5)
DEBUG = False


# --- The library menu, now that the tools have been tested. ---
def ask_number(prompt):
    """Ask for a whole number. isdigit() is True only when the text is all digits."""
    text = input(prompt).strip()
    while not text.isdigit():
        print("Please type a number.")
        text = input(prompt).strip()
    return int(text)


print()
print("========================================")
print("   LIBRARY MANAGEMENT SYSTEM")
print("========================================")
while True:
    print()
    print("1) Search   2) List by title   3) List newest first   4) Borrow   5) Return   6) Report   7) Quit")
    choice = input("Choose 1-7: ").strip()

    if choice == "1":
        text = input("Search for a title or an author: ").strip()
        found = search(books, text)
        if len(found) == 0:
            print("No books found.")
        else:
            show_books(found, "SEARCH RESULTS")
    elif choice == "2":
        show_books(sorted(books, key=get_title), "CATALOG, sorted by title")
    elif choice == "3":
        show_books(sorted(books, key=get_year, reverse=True), "CATALOG, newest first")
    elif choice == "4":
        who = ask_number("Member number (101 or 102): ")
        which = ask_number("Book id: ")
        ok, message = borrow_book(books, members, who, which)
        if ok:
            print("✅ " + message)
        else:
            print("⚠️  " + message)
    elif choice == "5":
        who = ask_number("Member number (101 or 102): ")
        which = ask_number("Book id: ")
        ok, message = return_book(books, members, who, which)
        if ok:
            print("✅ " + message)
        else:
            print("⚠️  " + message)
    elif choice == "6":
        print()
        print("REPORT")
        print("Books:", len(books), "| on the shelf:", len(available_books(books)), "| on loan:", len(books) - len(available_books(books)))
        for member_id in members:
            member = members[member_id]
            print(f"  {member['name']} ({member_id}) has {len(member['borrowed'])} book(s): {member['borrowed']}")
    elif choice == "7":
        print("Goodbye, and happy reading! 📚")
        break
    else:
        print("Please choose 1 to 7.")